In [1]:
from pymoo.indicators.hv import HV

import numpy as np
import pickle
import statistics
import pandas as pd

# Hypervolume:

In [30]:
# Name of the experiment
experiment_names = ['dtlz1', 'dtlz2', 'dtlz4', 'dtlz7'] # ['zdt1', 'zdt2', 'zdt3', 'zdt4', 'zdt6']
# Choosing the files
choose_global_best_attribution_type = [0] # [0, 1, 2, 3]
choose_Xr_pool_type = [0] # [0, 1, 2]
choose_DE_mutation_type = [0] # [0, 1, 2, 3, 4]

# Name of chosen files
file_names = [
        f"E{i+1}V{j+1}D{k+1}_{experiment_name}.pkl"
        for i in choose_global_best_attribution_type
        for j in choose_Xr_pool_type
        for k in choose_DE_mutation_type
        for experiment_name in experiment_names
    ]
num_dataset = len(file_names) # Number of dataset

# Take the results
results = []
for i in range(num_dataset):
  with open("../result/" + file_names[i], "rb") as f:
    results.append(pickle.load(f))

### Calculate the Hypervolume:

In [ ]:
# Create the hypervolume indicator
ref_point = np.array([11, 11])
indicator = HV(ref_point=ref_point)

# Create the pandas DataFrame columns
columns = ['Algorithm', 'Function', 'Mean HV', 'Std. Dev.', 'Min. HV', 'Max. HV', 'Combined HV']
# Store the datas
data = []
for i, fn in enumerate(file_names):
  HVs = []
  r = results[i]
  for j in range(len(results[i])-1):
    HVs.append(indicator(r[j+1]["F"]))
  if len(HVs) > 1:
    exp = fn.split('_', 1)
    alg = exp[0]
    func = (exp[1].split('.', 1))[0]
    data.append([alg, func.upper(), statistics.mean(HVs), statistics.stdev(HVs), min(HVs), max(HVs), indicator(results[i]['combined'][1])])
  else:
    exp = fn.split('_', 1)
    alg = exp[0]
    func = (exp[1].split('.', 1))[0]
    data.append([alg, func.upper(), statistics.mean(HVs), 0, min(HVs), max(HVs), indicator(results[i]['combined'][1])])

# Create the DataFrame
df = pd.DataFrame(data, columns=columns)
df